# Quoridor AI — N=4 Training (5×5, max^n)

**Group 501** | Colman College | DL Final Project

4-player AlphaZero-inspired agent using vector value heads + max^n MCTS.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells in order
3. If Colab disconnects, re-run from **Section 1** — training resumes from `latest.pt`

---
## 1. Environment Setup

In [ ]:
import os, sys

REPO_DIR = "./dl-quoridor"
BRANCH = "poc-4-player-comparison"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ReefKenig/dl-quoridor.git {REPO_DIR}

os.chdir(REPO_DIR)
!git fetch --all
!git checkout {BRANCH}
!git pull

%pip install -r requirements.txt -q
%pip install torch numpy -q

sys.path.append(os.getcwd())
print("Setup complete!")

In [ ]:
# Verify GPU
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Training will be slower (CPU-only MCTS).")

In [ ]:
# Checkpoint directory (persists across cell re-runs)
CHECKPOINT_DIR = "./checkpoints_mp_n4"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints: {CHECKPOINT_DIR}")

---
## 2. Smoke Test

In [ ]:
from src.env.quoridor_env_mp import QuoridorEnvMP
from src.model.network_mp import QuoridorModelMP
from src.mcts.mcts_maxn import MCTSMaxN, MCTSConfig
from src.mcts.self_play_mp import play_one_game
import numpy as np

N = 4
env = QuoridorEnvMP(board_size=5, num_players=N, max_turns=300, max_walls_per_player=4)
model = QuoridorModelMP(board_size=5, action_space_size=44, in_channels=3*N+3,
                        num_channels=64, num_res_blocks=4, num_players=N, device="auto")

mcts = MCTSMaxN(config=MCTSConfig(num_simulations=30), 
                evaluate_fn=lambda s: model.predict(env.state_to_tensor(s)),
                num_players=N)

samples, winner = play_one_game(env, mcts, N, max_moves=60)
print(f"Game: {len(samples)} samples, winner=seat {winner}")
print(f"Tensor: {samples[0][0].shape}, Policy: {samples[0][1].shape}, Value: {samples[0][2].shape}")
assert samples[0][0].shape == (5, 5, 15)  # 3*4+3 = 15 channels
assert samples[0][2].shape == (4,)         # vector value
print("\n✓ Smoke test passed")

---
## 3. Training

Re-run this cell after disconnect — it reloads from `latest.pt` if available.

In [ ]:
import logging
import time
import os
import sys
import numpy as np
import torch

# Force logging to stdout so Jupyter displays it inline
logging.basicConfig(level=logging.INFO, format="%(message)s",
                    stream=sys.stdout, force=True)
np.random.seed(0)
torch.manual_seed(0)

from src.env.quoridor_env_mp import QuoridorEnvMP
from src.model.network_mp import QuoridorModelMP
from src.mcts.training_mp import TrainingConfigMP, training_loop_mp

N = 4
env = QuoridorEnvMP(board_size=5, num_players=N, max_turns=300,
                    max_walls_per_player=4)

def make_model():
    return QuoridorModelMP(board_size=5, action_space_size=44,
                           in_channels=3*N+3, num_channels=64, num_res_blocks=4,
                           num_players=N, device="auto")

model = make_model()

# Resume is automatic — training_loop_mp reads meta.json + latest.pt from checkpoint_dir
cfg = TrainingConfigMP(
    num_players=N,
    num_iterations=20,
    games_per_iteration=40,
    mcts_simulations=100,
    batch_size=64,
    train_steps_per_iter=200,
    eval_games=80,
    eval_random_games=24,
    accept_margin=0.05,
    max_game_moves=300,
)

t0 = time.time()
history = training_loop_mp(env, model, make_model, cfg,
                           checkpoint_dir=CHECKPOINT_DIR)
total = time.time() - t0
print(f"\nDone! {len(history)} iterations in {total/3600:.1f}h")

---
## 4. Results

In [ ]:
import matplotlib.pyplot as plt

if history:
    iters = [h['iter'] for h in history]
    vs_rand = [h['win_vs_random'] * 100 for h in history]
    vs_best = [h['win_vs_best'] * 100 for h in history]
    loss_p = [h['loss_p'] for h in history]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].plot(iters, vs_rand, 'g-o', markersize=4)
    axes[0].axhline(y=25, color='gray', linestyle='--', label='fair share (25%)')
    axes[0].set_title('Win Rate vs Random (strength signal)')
    axes[0].set_ylabel('%')
    axes[0].set_ylim(0, 105)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(iters, vs_best, 'b-o', markersize=4)
    axes[1].axhline(y=30, color='orange', linestyle='--', label='accept threshold')
    axes[1].set_title('Win Rate vs Best (noisy at N=4)')
    axes[1].set_ylabel('%')
    axes[1].set_ylim(0, 60)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(iters, loss_p, 'r-o', markersize=4)
    axes[2].set_title('Policy Loss')
    axes[2].set_ylabel('Cross-Entropy')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('n4_training_curves.png', dpi=150)
    plt.show()
    
    print(f"\nFinal vs_rand: {vs_rand[-1]:.1f}% | Policy loss: {loss_p[-1]:.3f}")
    accepted = sum(1 for h in history if h['accepted'])
    print(f"Models accepted: {accepted}/{len(history)}")